### Import libraries
Spark functions for column operations; DeltaTable for merging into existing tables.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../../01_setup/utilities

### Utilities
Shared settings (catalog, schemas, S3 path) and a helper for full table names. Loaded by every pipeline notebook with `%run`.

### Notebook parameters, paths and table names
New files arrive in `landing`; after loading they are moved to `processed` so they are never loaded twice.

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"{s3_base_path}/{data_source}"
landing_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"

bronze_table = table_name(bronze_schema, data_source)
silver_table = table_name(silver_schema, data_source)
gold_table = table_name(gold_schema, f"sb_fact_{data_source}")
parent_fact_table = table_name(gold_schema, "fact_orders")

print("Landing path:  ", landing_path)
print("Processed path:", processed_path)
print("Tables:", bronze_table, silver_table, gold_table, parent_fact_table)

Landing path:   s3://sportsbar-data-lake-vkt/orders/landing
Processed path: s3://sportsbar-data-lake-vkt/orders/processed
Tables: fmcg.bronze.orders fmcg.silver.orders fmcg.gold.sb_fact_orders fmcg.gold.fact_orders


## Bronze

### Read new files from landing
Stops the notebook cleanly if there are no new files, instead of failing with an error.

In [0]:
landing_files = [f for f in dbutils.fs.ls(landing_path) if f.name.endswith(".csv")]
print("Files in landing:", len(landing_files))

if not landing_files:
    dbutils.notebook.exit("No new files in landing – nothing to load.")

df = (
    spark.read.options(header=True, inferSchema=True)
    .csv(f"{landing_path}/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total rows:", df.count())
df.printSchema()
display(df.limit(5))

Files in landing: 151
Total rows: 51810
root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FOCT62720602,"Tuesday, September 30, 2025",ABC987,25891301,71.0,2026-11-04T16:00:40.923Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891502,125.0,2026-11-04T16:00:40.923Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891403,462.0,2026-11-04T16:00:40.923Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",INVALID,25891601,133.0,2026-11-04T16:00:40.923Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891602,79.0,2026-11-04T16:00:40.923Z,orders_2025_09_30.csv,41446


### Append to bronze
`append` adds new rows and keeps everything loaded before, so bronze holds the full history of raw files.

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("append")
    .saveAsTable(bronze_table)
)

### Move loaded files from landing to processed
Runs only after bronze is saved, so a file is never moved without being loaded.

In [0]:
for f in landing_files:
    dbutils.fs.mv(f.path, f"{processed_path}/{f.name}")

print("Files moved to processed:", len(landing_files))

Files moved to processed: 151


## Silver

### Load bronze data

In [0]:
df_orders = spark.table(bronze_table)
print("Rows in bronze:", df_orders.count())
display(df_orders.limit(5))

Rows in bronze: 51810


order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025/07/01,789320,25891203,150.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744
FJUL33320501,2025/07/01,789320,25891301,46.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744
FJUL33320501,01-07-2025,789320,25891403,null,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891201,354.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891501,249.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744


### 1. Remove orders without a quantity

In [0]:
print("Rows without quantity:", df_orders.filter(F.col("order_qty").isNull()).count())
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

Rows without quantity: 2607


### 2. Clean customer_id
Keeps numeric IDs; anything else becomes 999999 so the order is not lost.

In [0]:
customer_text = F.col("customer_id").cast("string")

df_orders = df_orders.withColumn(
    "customer_id",
    F.when(customer_text.rlike("^[0-9]+$"), customer_text).otherwise(F.lit("999999"))
)

print("Orders with invalid customer_id:", df_orders.filter(F.col("customer_id") == "999999").count())

Orders with invalid customer_id: 4943


### 3. Standardise the order date
Removes weekday names ("Tuesday, July 01, 2025" → "July 01, 2025"), then tries each known date format.

In [0]:
date_text = F.regexp_replace(F.col("order_placement_date").cast("string"), r"^[A-Za-z]+,\s*", "")

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(date_text, "yyyy/MM/dd"),
        F.try_to_date(date_text, "dd-MM-yyyy"),
        F.try_to_date(date_text, "dd/MM/yyyy"),
        F.try_to_date(date_text, "yyyy-MM-dd"),
        F.try_to_date(date_text, "MMMM dd, yyyy"),
    )
)

print("Unreadable dates:", df_orders.filter(F.col("order_placement_date").isNull()).count())

Unreadable dates: 0


### 4. Remove duplicate orders and convert product_id to text

In [0]:
rows_before = df_orders.count()
df_orders = df_orders.dropDuplicates(
    ["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"]
)
print("Duplicates removed:", rows_before - df_orders.count())

df_orders = df_orders.withColumn("product_id", F.col("product_id").cast("string"))

Duplicates removed: 3616


### Check the date range
Expected: 2025-07-01 to 2025-11-30. Days with orders shows whether any days are missing.

In [0]:
df_orders.agg(
    F.min("order_placement_date").alias("first_date"),
    F.max("order_placement_date").alias("last_date"),
    F.countDistinct("order_placement_date").alias("days_with_orders"),
).show()

+----------+----------+----------------+
|first_date| last_date|days_with_orders|
+----------+----------+----------------+
|2025-07-01|2025-11-30|             151|
+----------+----------+----------------+



In [0]:
all_days = spark.sql("SELECT explode(sequence(DATE'2025-07-01', DATE'2025-11-30')) AS day")
order_days = df_orders.select(F.col("order_placement_date").alias("day")).distinct()
display(all_days.join(order_days, "day", "left_anti"))

day
2025-08-31
2025-10-31


### 5. Attach product_code
Looks up each product_id in silver products. Orders without a matching product are shown first, then dropped.

In [0]:
df_products = spark.table(table_name(silver_schema, "products")).select("product_id", "product_code")

df_unmatched = df_orders.join(df_products, "product_id", "left_anti")
print("Orders without a matching product:", df_unmatched.count())
display(df_unmatched.groupBy("product_id").count().orderBy("product_id"))

df_joined = df_orders.join(df_products, "product_id", "inner")
display(df_joined.limit(5))

Orders without a matching product: 4776


product_id,count
25891502,2435
77777777,816
88888888,757
99999999,768


product_id,order_id,order_placement_date,customer_id,order_qty,read_timestamp,file_name,file_size,product_code
25891103,FJUL32202603,2025-07-01,789202,370.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
25891602,FJUL33321602,2025-07-01,789321,74.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
25891102,FJUL34303602,2025-07-01,789303,317.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
25891402,FJUL34622602,2025-07-01,789622,439.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49
25891303,FJUL33402303,2025-07-01,789402,63.0,2026-11-04T16:06:04.025Z,orders_2025_07_01.csv,20744,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f


### Save to silver
First run creates the table; later runs merge, so re-loading the same orders never duplicates them.

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
        df_joined.write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
    print("Silver table created")
else:
    (
        DeltaTable.forName(spark, silver_table).alias("target")
        .merge(
            df_joined.alias("source"),
            """target.order_placement_date = source.order_placement_date
               AND target.order_id = source.order_id
               AND target.product_code = source.product_code
               AND target.customer_id = source.customer_id""",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver table merged")

print("Rows in silver:", spark.table(silver_table).count())

Silver table created
Rows in silver: 40811


## Gold

### Rename to business names and save to gold
Daily SportsBar orders, saved as `sb_fact_orders`.

In [0]:
df_gold = spark.table(silver_table).select(
    "order_id",
    F.col("order_placement_date").alias("date"),
    F.col("customer_id").alias("customer_code"),
    "product_code",
    "product_id",
    F.col("order_qty").alias("sold_quantity"),
)

if not spark.catalog.tableExists(gold_table):
    (
        df_gold.write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )
    print("Gold table created")
else:
    (
        DeltaTable.forName(spark, gold_table).alias("target")
        .merge(
            df_gold.alias("source"),
            """target.date = source.date
               AND target.order_id = source.order_id
               AND target.product_code = source.product_code
               AND target.customer_code = source.customer_code""",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Gold table merged")

print("Rows in gold:", spark.table(gold_table).count())

Gold table created
Rows in gold: 40811


## Merge into parent

### Check Atlikon's fact table level
Atlikon stores one row per month (always the 1st). Should show only day 1.

In [0]:
display(
    spark.table(parent_fact_table)
    .select(F.dayofmonth("date").alias("day_of_month"))
    .distinct()
)

day_of_month
1


### Sum daily orders to monthly level
One row per month, product and customer, dated the 1st of the month – the same level as Atlikon.

In [0]:
target_types = dict(spark.table(parent_fact_table).dtypes)

df_monthly = (
    spark.table(gold_table)
    .withColumn("date", F.trunc("date", "MM"))
    .groupBy("date", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumn("sold_quantity", F.col("sold_quantity").cast(target_types["sold_quantity"]))
    .withColumn("customer_code", F.col("customer_code").cast(target_types["customer_code"]))
)

print("Daily rows:  ", spark.table(gold_table).count())
print("Monthly rows:", df_monthly.count())
display(df_monthly.limit(5))

Daily rows:   40811
Monthly rows: 3060


date,product_code,customer_code,sold_quantity
2025-07-01,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,789202,2944
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789321,1742
2025-07-01,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789303,4929
2025-07-01,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,789622,3810
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789402,960


### Check customers exist in dim_customers
Orders for customer codes missing from `dim_customers` (e.g. 999999) won't show a customer name in the dashboard.

In [0]:
dim_customer_codes = spark.table(table_name(gold_schema, "dim_customers")).select("customer_code")
missing_customers = df_monthly.join(dim_customer_codes, "customer_code", "left_anti")
print("Monthly rows with unknown customer:", missing_customers.count())
display(missing_customers.groupBy("customer_code").agg(F.sum("sold_quantity").alias("units")))

Monthly rows with unknown customer: 85


customer_code,units
999999,949848


In [0]:
total_units = df_monthly.agg(F.sum("sold_quantity")).first()[0]
print(f"Unknown customer share: {949848 / total_units:.1%}")

Unknown customer share: 10.1%


### Merge into Atlikon's fact_orders
Matches on month, product and customer. Safe to re-run.

In [0]:
rows_before = spark.table(parent_fact_table).count()

(
    DeltaTable.forName(spark, parent_fact_table).alias("target")
    .merge(
        df_monthly.alias("source"),
        """target.date = source.date
           AND target.product_code = source.product_code
           AND target.customer_code = source.customer_code""",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

rows_after = spark.table(parent_fact_table).count()
print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Rows added: ", rows_after - rows_before)

Rows before: 93055
Rows after:  96115
Rows added:  3060
